In [ ]:
from datetime import datetime
from typing import Optional
from uuid import uuid4

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, EmailStr, Field, field_serializer, UUID4

app = FastAPI()

Aqui, além da importação de tipos nativos do Python (`datetime` e `uuid4`) remos a importação da FastAPI. A linha `app = FastAPI()` cria o servidor web e cenquanto o pydantic fica reponsáevel por validar os dados recebidos via HTTP.

In [ ]:
class User(BaseModel):
    model_config = {
        "extra": "forbid",
    }
    __users__ = []
    
    name: str = Field(..., description="Name of the user")
    email: EmailStr = Field(..., description="Email address of the user")
    
    friends: list[UUID4] = Field(default_factory=list, max_items=500)
    blocked: list[UUID4] = Field(default_factory=list, max_items=500)
    
    signup_ts: Optional[datetime] = Field(default_factory=datetime.now, kw_only=True)
    id: UUID4 = Field(default_factory=uuid4, kw_only=True)

    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        return str(id)

Aqui, a variável `__users__` é um vetor de users mockado, já que não temos banco de dados real aqui. A classe implementa os dados referentes ao User usando `Field()`.

In [ ]:
@app.get("/users", response_model=list[User])
async def get_users() -> list[User]:
    return list(User.__users__)

@app.post("/users", response_model=User)
async def create_user(user: User):
    User.__users__.append(user)
    return user

Aqui, são criados os endpoints. Um get que retorna a lista de usuários e um post que cria um usuário.

Ao declarar `async def create_user(user: User)`, o FastAPI intercepta o texto JSON que chegou da internet, entrega pro Pydantic, e o Pydantic faz todo o trabalho de validação. Se os dados estiverem perfeitos, a função recebe a instância já montada. Se estiverem errados, o FastAPI devolve o erro para o cliente sem que a função chegue sequer a ser ativada.

In [ ]:
@app.get("/users/{user_id}", response_model=User)
async def get_user(user_id: UUID4) -> User | JSONResponse:
    try:
        return next((user for user in User.__users__ if user.id == user_id))
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "User not found"})

Rota GET que busca um usuário pelo ID. Por ser um id `UUID4`, todos os IDs são únicos e do tipo `UUID4`, logo, IDs não válidos retornam um erro pelo Pydantic.

In [ ]:
def main() -> None:
    with TestClient(app) as client:
        for i in range(5):
            response = client.post(
                "/users",
                json={"name": f"User {i}", "email": f"example{i}@arjancodes.com"},
            )
            assert response.status_code == 200
            assert response.json()["name"] == f"User {i}", (
                "The name of the user should be User {i}"
            )
            assert response.json()["id"], "The user should have an id"

            user = User.model_validate(response.json())
            assert str(user.id) == response.json()["id"], "The id should be the same"
            assert user.signup_ts, "The signup timestamp should be set"
            assert user.friends == [], "The friends list should be empty"
            assert user.blocked == [], "The blocked list should be empty"

        response = client.get("/users")
        assert response.status_code == 200, "Response code should be 200"
        assert len(response.json()) == 5, "There should be 5 users"

        response = client.post(
            "/users", json={"name": "User 5", "email": "example5@arjancodes.com"}
        )
        assert response.status_code == 200
        assert response.json()["name"] == "User 5", (
            "The name of the user should be User 5"
        )
        assert response.json()["id"], "The user should have an id"

        user = User.model_validate(response.json())
        assert str(user.id) == response.json()["id"], "The id should be the same"
        assert user.signup_ts, "The signup timestamp should be set"
        assert user.friends == [], "The friends list should be empty"
        assert user.blocked == [], "The blocked list should be empty"

        response = client.get(f"/users/{response.json()['id']}")
        assert response.status_code == 200
        assert response.json()["name"] == "User 5", (
            "This should be the newly created user"
        )

        response = client.get(f"/users/{uuid4()}")
        assert response.status_code == 404
        assert response.json()["message"] == "User not found", (
            "We technically should not find this user"
        )

        response = client.post("/users", json={"name": "User 6", "email": "wrong"})
        assert response.status_code == 422, "The email address is should be invalid"


if __name__ == "__main__":
    main()

A `main` usa `TestClient` para simular as requisições HTTP. Ele atesta que os usuários são criados, com ID e Data sendo gerados sozinhos e a lista de amigos iniciando vazia.

na ultimas linhas, é feito um POST de um user cujo email está propositalmente errado. Como o FastAPI está usando o Pydantic, o servidor não quebra. Em vez disso, ele responde automaticamente com o **Status HTTP 422**.